In [ ]:
import csv
import importlib.util
import json
import math
import os
import shutil
import tarfile
import tempfile
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PRITHVI_REPO    = "ibm-nasa-geospatial/Prithvi-EO-1.0-100M"
PRITHVI_WEIGHTS = "Prithvi_EO_V1_100M.pt"
PRITHVI_CFG     = "config.json"
PRITHVI_SRC     = "prithvi_mae.py"

EMBED_DIM        = 768
PATCH_SIZE       = 16
TILE_SIZE        = 224
NUM_PATCHES      = (TILE_SIZE // PATCH_SIZE) ** 2
HLS_CHANNELS     = 6
SAR_CHANNELS     = 4
SPECTRAL_QUERIES = 8
DECODER_DIM      = 256
MAMBA_D_STATE    = 16
MAMBA_D_CONV     = 4
SCATTER_COMPS    = 3
NUM_FRAMES       = 1

DATASET_REPO    = "ibm-nasa-geospatial/multi-temporal-crop-classification"
DATASET_ARCHIVE = "validation_chips.tgz"
NUM_TEST_CHIPS  = 6

HLS_MEAN = np.array([775.2290211032589, 1080.992780391705, 1228.5855250417867,
                      2497.2022620507532, 2204.2139147975554, 1610.8324823273745], dtype=np.float32)
HLS_STD  = np.array([1281.526139861424, 1270.0297974547493, 1399.4802505642526,
                      1368.3446143747644, 1291.6764008585435, 1154.505683480695], dtype=np.float32)

TILE_STRIDE = 196
BATCH_SIZE  = 4

MAX_CHL_UG_CM2      = 80.0
MAX_N_PERCENT       = 4.5
MAX_BIOMASS_MGHA    = 250.0
HEALTHY_BIOMASS_REF = 80.0

VEG_NDVI_THRESH     = 0.20
STRESS_THRESHOLD    = 0.35
BIOMASS_LOSS_THRESH = 0.25

DATA_DIR   = "./data"
OUTPUT_DIR = "./output"
ABLATION   = True

CDL_CLASSES = {
    0: "Background",   1: "Corn",        2: "Cotton",
    3: "Rice",         4: "Sorghum",     5: "Soybeans",
    6: "Sunflower",    7: "Peanuts",     8: "Tobacco",
    9: "Sweet Corn",  10: "Pop. Corn",  11: "Mint",
   12: "Winter Wheat",
}

_PRITHVI_CFG_DROP = frozenset({"mask_ratio", "bands", "mean", "std", "origin_url", "paper_ids"})


def _fmt(label, value, width=42):
    return f"  {label:<{width}}{value}"

def _section(title):
    w = 72
    print(f"\n{'━'*w}")
    print(f"  {title}")
    print(f"{'━'*w}")


def build_hls_bands(image: np.ndarray) -> np.ndarray:
    C = image.shape[0]
    if C >= 6:
        return image[:6].copy().astype(np.float32)
    if C == 3:
        R, G, B = image[0], image[1], image[2]
        RE1 = R * 0.55 + G * 0.30 + B * 0.15
        RE2 = R * 0.60 + G * 0.25 + B * 0.15
        RE3 = R * 0.50 + G * 0.30 + B * 0.20
        return np.stack([B, G, R, RE1, RE2, RE3]).astype(np.float32)
    if C == 1:
        return np.tile(image, (6, 1, 1)).astype(np.float32)
    pad = np.zeros((6 - C, image.shape[1], image.shape[2]), dtype=np.float32)
    return np.concatenate([image[:C], pad], axis=0).astype(np.float32)


def normalize_hls(hls: np.ndarray) -> np.ndarray:
    out = hls.copy()
    for i in range(6):
        out[i] = (out[i] - HLS_MEAN[i]) / (HLS_STD[i] + 1e-8)
    return out


def build_sar_proxy(spatial_shape: Tuple[int, int]) -> np.ndarray:
    H, W = spatial_shape
    rng  = np.random.default_rng(seed=0)
    vv   = rng.uniform(-15.0, -5.0,  (H, W)).astype(np.float32)
    vh   = (vv - rng.uniform(5.0, 12.0, (H, W))).astype(np.float32)
    rvi  = (4.0 * 10 ** (vh / 10.0)) / (10 ** (vv / 10.0) + 10 ** (vh / 10.0) + 1e-8)
    coh  = rng.uniform(0.3, 0.9, (H, W)).astype(np.float32)
    return np.stack([vv, vh, rvi.astype(np.float32), coh], axis=0)


def compute_spectral_indices(hls: np.ndarray) -> np.ndarray:
    B02, B03, B04 = hls[0], hls[1], hls[2]
    B05, B06, B07 = hls[3], hls[4], hls[5]
    eps = 1e-8
    NIR, Red = B07, B04
    ndvi  = (NIR - Red)  / (NIR + Red  + eps)
    ndre  = (NIR - B05)  / (NIR + B05  + eps)
    evi   = 2.5 * (NIR - Red) / (NIR + 6 * Red - 7.5 * B02 + 1 + eps)
    cire  = (NIR / (B05 + eps)) - 1.0
    ndwi  = (B03 - NIR) / (B03 + NIR + eps)
    mndwi = (B03 - B06) / (B03 + B06 + eps)
    savi  = 1.5 * (NIR - Red) / (NIR + Red + 0.5 + eps)
    rvi   = NIR / (Red + eps)
    return np.stack([ndvi, ndre, evi, cire, ndwi, mndwi, savi, rvi]).astype(np.float32)


def tile_image(hls_norm, sar, indices, tile_size=TILE_SIZE, stride=TILE_STRIDE):
    _, H, W = hls_norm.shape
    tiles = []
    y = 0
    while True:
        ey = min(y + tile_size, H)
        sy = max(0, ey - tile_size)
        x  = 0
        while True:
            ex = min(x + tile_size, W)
            sx = max(0, ex - tile_size)
            ch, cw = ey - sy, ex - sx
            hls_t = np.zeros((hls_norm.shape[0], tile_size, tile_size), dtype=np.float32)
            sar_t = np.zeros((sar.shape[0],      tile_size, tile_size), dtype=np.float32)
            idx_t = np.zeros((indices.shape[0],  tile_size, tile_size), dtype=np.float32)
            hls_t[:, :ch, :cw] = hls_norm[:, sy:ey, sx:ex]
            sar_t[:, :ch, :cw] = sar[:,     sy:ey, sx:ex]
            idx_t[:, :ch, :cw] = indices[:, sy:ey, sx:ex]
            tiles.append({"hls": hls_t, "sar": sar_t, "indices": idx_t,
                          "y_start": sy, "x_start": sx, "y_end": ey, "x_end": ex,
                          "valid_h": ch, "valid_w": cw})
            if ex == W: break
            x += stride
        if ey == H: break
        y += stride
    return tiles


def _cbg(ic, oc, k=3, s=1, p=1):
    return nn.Sequential(nn.Conv2d(ic, oc, k, s, p, bias=False), nn.BatchNorm2d(oc), nn.GELU())

def _dcbg(ic, oc, k=4, s=2, p=1):
    return nn.Sequential(nn.ConvTranspose2d(ic, oc, k, s, p, bias=False), nn.BatchNorm2d(oc), nn.GELU())


@dataclass
class AblationConfig:
    name:               str
    zero_sar:           bool = False
    zero_indices:       bool = False
    zero_nir:           bool = False
    zero_swir:          bool = False
    zero_red_edge:      bool = False
    disable_scad:       bool = False
    disable_pmfd:       bool = False
    disable_ssm:        bool = False
    disable_scatter:    bool = False
    disable_cross_attn: bool = False


ABLATION_VARIANTS: List[AblationConfig] = [
    AblationConfig("Full model (baseline)"),
    AblationConfig("-SAR input",           zero_sar=True),
    AblationConfig("-Spectral indices",    zero_indices=True),
    AblationConfig("-NIR proxy (B07)",     zero_nir=True),
    AblationConfig("-Red-edge (B05-B07)",  zero_swir=True),
    AblationConfig("-Red band (B04)",      zero_red_edge=True),
    AblationConfig("-SCAD decoder",        disable_scad=True),
    AblationConfig("-PMFD decoder",        disable_pmfd=True),
    AblationConfig("-SSM (Mamba gate)",    disable_ssm=True),
    AblationConfig("-Scatter decomp",      disable_scatter=True),
    AblationConfig("-Cross-attention",     disable_cross_attn=True),
]


@dataclass
class ModelOutput:
    chl_map:      torch.Tensor
    nitro_map:    torch.Tensor
    biomass_map:  torch.Tensor
    loss_map:     torch.Tensor
    variant_name: str = "Full model (baseline)"


@dataclass
class AblationOutput:
    variants: Dict[str, ModelOutput] = field(default_factory=dict)

    def delta_vs_baseline(self, key):
        b, v = self.variants["Full model (baseline)"], self.variants[key]
        def _d(a, bb): return float((a - bb).abs().mean().item())
        return {"d_chl": _d(v.chl_map, b.chl_map), "d_nitro": _d(v.nitro_map, b.nitro_map),
                "d_biomass": _d(v.biomass_map, b.biomass_map), "d_loss": _d(v.loss_map, b.loss_map)}

    def summary_table(self):
        BASE  = "Full model (baseline)"
        W     = 78
        lines = [f"\n{'━'*W}",
                 f"  {'ABLATION STUDY  —  Mean Absolute Delta vs Baseline':^{W-4}}",
                 f"{'━'*W}",
                 f"  {'Variant':<32} {'Δ CHL':>8} {'Δ Nitro':>8} {'Δ Biomass':>10} {'Δ Loss':>8}",
                 f"{'─'*W}"]
        for name, out in self.variants.items():
            if name == BASE:
                lines.append(f"  {name:<32}  ← baseline")
                continue
            d = self.delta_vs_baseline(name)
            lines.append(f"  {name:<32} {d['d_chl']:>8.4f} {d['d_nitro']:>8.4f}"
                         f" {d['d_biomass']:>10.4f} {d['d_loss']:>8.4f}")
        lines += [f"{'━'*W}",
                  f"  Δ = mean |pred_ablated − pred_baseline|  (0–1 scale)  ·  higher = more important",
                  f"{'━'*W}"]
        return "\n".join(lines)


class PrithviBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self._inner    = None
        self._strategy = "none"
        self._try_automodel()
        if self._inner is None:
            self._try_pt_download()
        if self._inner is None:
            raise RuntimeError("Prithvi-EO-1.0-100M failed to load via both AutoModel and .pt strategies.")

    def _try_automodel(self):
        try:
            from transformers import AutoModel
            print(f"  [Backbone] Trying AutoModel.from_pretrained …")
            m = AutoModel.from_pretrained(PRITHVI_REPO, trust_remote_code=True)
            self._inner    = m
            self._strategy = "automodel"
            n = sum(p.numel() for p in m.parameters())
            print(f"  [Backbone] ✓ AutoModel loaded  ({n:,} params)")
        except Exception as e:
            print(f"  [Backbone] ✗ AutoModel failed: {e}")

    def _try_pt_download(self):
        try:
            from huggingface_hub import hf_hub_download
            hf_token = os.getenv("HF_TOKEN", None)
            print(f"  [Backbone] Trying hf_hub_download ({PRITHVI_WEIGHTS}) …")
            ckpt_path = hf_hub_download(PRITHVI_REPO, PRITHVI_WEIGHTS, token=hf_token)
            cfg_path  = hf_hub_download(PRITHVI_REPO, PRITHVI_CFG,token=hf_token)
            src_path  = hf_hub_download(PRITHVI_REPO, PRITHVI_SRC,token=hf_token)
            try:
                local_src = Path(__file__).parent / "prithvi_mae.py"
            except NameError:
                local_src = Path(os.getcwd()) / "prithvi_mae.py"
            if not local_src.exists():
                shutil.copy(src_path, local_src)
            spec = importlib.util.spec_from_file_location("prithvi_mae", str(local_src))
            mod  = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            with open(cfg_path) as f:
                raw_cfg = json.load(f)
            pcfg    = raw_cfg.get("pretrained_cfg", raw_cfg)
            init_kw = {k: v for k, v in pcfg.items() if k not in _PRITHVI_CFG_DROP}
            init_kw["num_frames"] = NUM_FRAMES
            model = None
            for cls_name in ("PrithviViT", "PrithviMAE"):
                if hasattr(mod, cls_name):
                    try:
                        model = getattr(mod, cls_name)(**init_kw)
                        print(f"  [Backbone] Instantiated {cls_name}")
                        break
                    except Exception as e:
                        print(f"  [Backbone] {cls_name} failed: {e}")
            if model is None:
                raise RuntimeError("Neither PrithviViT nor PrithviMAE found.")
            state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            if isinstance(state, dict) and "model" in state and isinstance(state["model"], dict):
                state = state["model"]
            state        = {k: v for k, v in state.items() if "pos_embed" not in k}
            miss, unexp  = model.load_state_dict(state, strict=False)
            model.eval()
            self._inner    = model
            self._strategy = "pt_direct"
            n = sum(p.numel() for p in model.parameters())
            print(f"  [Backbone] ✓ {type(model).__name__} ready  ({n:,} params)  "
                  f"missing={len(miss)}  unexpected={len(unexp)}")
        except Exception as e:
            print(f"  [Backbone] ✗ .pt load failed: {e}")

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        if self._strategy == "automodel":
            with torch.no_grad():
                out = self._inner(pixel_values=x)
            hs = getattr(out, "last_hidden_state", None) or out[0]
            return hs[:, 1:, :] if hs.shape[1] == NUM_PATCHES + 1 else hs
        if self._strategy == "pt_direct":
            x5 = x.unsqueeze(2)
            with torch.no_grad():
                if hasattr(self._inner, "forward_encoder"):
                    latent, _, _ = self._inner.forward_encoder(x5, mask_ratio=0.0)
                else:
                    out    = self._inner(x5)
                    latent = out[0] if isinstance(out, (tuple, list)) else out
            return latent[:, 1:, :]
        raise RuntimeError(f"Unknown backbone strategy: {self._strategy}")

    def forward(self, x):
        return self.encode(x)


class BandRatioQueryGenerator(nn.Module):
    def __init__(self, ni=8, nq=SPECTRAL_QUERIES, d=EMBED_DIM):
        super().__init__()
        self.proj = nn.Linear(ni, d)
        self.gen  = nn.Sequential(nn.Linear(d, d * 2), nn.GELU(), nn.Linear(d * 2, nq * d))
        self.nq, self.d = nq, d

    def forward(self, idx):
        pooled = idx.mean(dim=[2, 3])
        return self.gen(self.proj(pooled)).view(pooled.shape[0], self.nq, self.d)


class SpectralCrossAttentionDecoder(nn.Module):
    def __init__(self, d=EMBED_DIM, dd=DECODER_DIM, nq=SPECTRAL_QUERIES):
        super().__init__()
        self.qgen    = BandRatioQueryGenerator(8, nq, d)
        self.xattn   = nn.MultiheadAttention(d, 8, batch_first=True)
        self.qnorm   = nn.LayerNorm(d)
        self.tnorm   = nn.LayerNorm(d)
        self.to_feat = nn.Linear(d, dd)
        self.up1 = _dcbg(dd,    dd);     self.r1 = _cbg(dd,    dd)
        self.up2 = _dcbg(dd,    dd//2);  self.r2 = _cbg(dd//2, dd//2)
        self.up3 = _dcbg(dd//2, dd//4);  self.r3 = _cbg(dd//4, dd//4)
        self.up4 = _dcbg(dd//4, dd//8);  self.r4 = _cbg(dd//8, dd//8)
        fc = dd // 8
        self.chl_head   = nn.Sequential(_cbg(fc, 32), nn.Conv2d(32, 1, 1), nn.Sigmoid())
        self.nitro_head = nn.Sequential(_cbg(fc, 32), nn.Conv2d(32, 1, 1), nn.Sigmoid())

    def forward(self, tokens, indices, disable_cross_attn=False):
        B, N, D = tokens.shape
        side    = int(math.sqrt(N))
        tn      = self.tnorm(tokens)
        if not disable_cross_attn:
            q     = self.qnorm(self.qgen(indices))
            tn, _ = self.xattn(q, tn, tn)
            tn    = self.tnorm(tokens + F.linear(
                tn.mean(1, keepdim=True).expand_as(tokens),
                torch.eye(D, device=tokens.device)))
        feat = self.to_feat(tn).transpose(1, 2).reshape(B, -1, side, side)
        feat = self.r1(self.up1(feat))
        feat = self.r2(self.up2(feat))
        feat = self.r3(self.up3(feat))
        feat = self.r4(self.up4(feat))
        return self.chl_head(feat), self.nitro_head(feat)


class ScatteringDecomposition(nn.Module):
    def __init__(self, ic=SAR_CHANNELS, oc=SCATTER_COMPS):
        super().__init__()
        self.net = nn.Sequential(_cbg(ic, 64), _cbg(64, 64),
                                 nn.Conv2d(64, oc, 1), nn.Softmax(dim=1))
        self.oc  = oc

    def forward(self, sar, disable_scatter=False):
        if disable_scatter:
            B, _, H, W = sar.shape
            return torch.full((B, self.oc, H, W), 1.0 / self.oc, device=sar.device, dtype=sar.dtype)
        return self.net(sar)


class SelectiveStateScan(nn.Module):
    def __init__(self, d=EMBED_DIM, S=MAMBA_D_STATE, dc=MAMBA_D_CONV):
        super().__init__()
        self.d, self.S = d, S
        self.in_proj   = nn.Linear(d, d * 2)
        self.conv1d    = nn.Conv1d(d, d, dc, padding=dc - 1, groups=d)
        self.x_proj    = nn.Linear(d, S + S + 1)
        self.dt_proj   = nn.Linear(1, S)
        self.A_log     = nn.Parameter(torch.log(torch.arange(1, S + 1).float()))
        self.D         = nn.Parameter(torch.ones(d))
        self.out_proj  = nn.Linear(d, d)
        self.ratio_gate= nn.Sequential(nn.Linear(1, d), nn.Sigmoid())
        self.bypass    = nn.Linear(d, d)

    def forward(self, tokens, sar_ratio=None, disable_ssm=False):
        if disable_ssm:
            return self.bypass(tokens)
        B, N, D = tokens.shape
        S       = self.S
        xz      = self.in_proj(tokens)
        x, z    = xz.chunk(2, dim=-1)
        xc      = self.conv1d(x.transpose(1, 2))[:, :, :N].transpose(1, 2)
        xc      = F.silu(xc)
        dBC     = self.x_proj(xc)
        dtr, Bm, Cm = dBC.split([1, S, S], dim=-1)
        dt      = F.softplus(self.dt_proj(dtr))
        A       = -torch.exp(self.A_log.float())
        h       = torch.zeros(B, S, D, device=tokens.device, dtype=tokens.dtype)
        ys      = []
        for i in range(N):
            dA  = torch.exp(dt[:, i, :] * A.unsqueeze(0))
            dBu = (dt[:, i, :].unsqueeze(-1) * Bm[:, i, :].unsqueeze(-1) * xc[:, i, :].unsqueeze(1))
            h   = h * dA.unsqueeze(-1) + dBu
            ys.append((h * Cm[:, i, :].unsqueeze(-1)).sum(1))
        y = torch.stack(ys, dim=1) + xc * self.D.unsqueeze(0).unsqueeze(0)
        if sar_ratio is not None:
            y = y * self.ratio_gate(sar_ratio.unsqueeze(-1))
        return self.out_proj(y * F.silu(z))


class PolarimetricFusionDecoder(nn.Module):
    def __init__(self, d=EMBED_DIM, dd=DECODER_DIM, sc=SAR_CHANNELS):
        super().__init__()
        self.sar_enc = nn.Sequential(_cbg(sc, 64), _cbg(64, 128), _cbg(128, dd))
        self.scatter = ScatteringDecomposition(sc, SCATTER_COMPS)
        self.sc_fuse = _cbg(SCATTER_COMPS, dd // 4)
        self.ssm     = SelectiveStateScan(d, MAMBA_D_STATE, MAMBA_D_CONV)
        self.to_feat = nn.Linear(d, dd)
        self.xattn   = nn.MultiheadAttention(dd, 8, batch_first=True)
        self.xnorm   = nn.LayerNorm(dd)
        self.up1 = _dcbg(dd + dd // 4, dd);  self.r1 = _cbg(dd,    dd)
        self.up2 = _dcbg(dd,           dd // 2); self.r2 = _cbg(dd//2, dd//2)
        self.up3 = _dcbg(dd // 2,      dd // 4); self.r3 = _cbg(dd//4, dd//4)
        self.up4 = _dcbg(dd // 4,      dd // 8); self.r4 = _cbg(dd//8, dd//8)
        fc = dd // 8
        self.bio_head  = nn.Sequential(_cbg(fc, 32), nn.Conv2d(32, 1, 1), nn.Sigmoid())
        self.loss_head = nn.Sequential(_cbg(fc, 32), nn.Conv2d(32, 1, 1), nn.Sigmoid())

    def _sar_ratio_per_patch(self, sar_maps, side):
        p = F.adaptive_avg_pool2d(sar_maps, (side, side)).flatten(2).transpose(1, 2)
        return p[:, :, 0] / (p[:, :, 1].abs() + 1e-8)

    def forward(self, tokens, sar, disable_ssm=False, disable_scatter=False):
        B    = tokens.shape[0]
        side = int(math.sqrt(tokens.shape[1]))
        sm   = self.sar_enc(sar)
        sc   = self.scatter(sar, disable_scatter=disable_scatter)
        scf  = self.sc_fuse(sc)
        rat  = self._sar_ratio_per_patch(sm, side)
        sca  = self.ssm(tokens, rat, disable_ssm=disable_ssm)
        tf   = self.to_feat(sca)
        ss   = F.adaptive_avg_pool2d(sm, (side, side)).flatten(2).transpose(1, 2)
        fn   = self.xnorm(tf)
        fus, _ = self.xattn(fn, self.xnorm(ss), self.xnorm(ss))
        feat   = fus.transpose(1, 2).reshape(B, -1, side, side)
        scd    = F.interpolate(scf, (side, side), mode="bilinear", align_corners=False)
        feat   = torch.cat([feat, scd], dim=1)
        feat   = self.r1(self.up1(feat))
        feat   = self.r2(self.up2(feat))
        feat   = self.r3(self.up3(feat))
        feat   = self.r4(self.up4(feat))
        return self.bio_head(feat), self.loss_head(feat)


class DualTaskCropHealthModel(nn.Module):
    def __init__(self):
        super().__init__()
        _section("MODEL INIT  ·  Prithvi-EO-1.0-100M + SCAD + PMFD")
        self.backbone = PrithviBackbone()
        self.scad     = SpectralCrossAttentionDecoder(EMBED_DIM, DECODER_DIM, SPECTRAL_QUERIES)
        self.pmfd     = PolarimetricFusionDecoder(EMBED_DIM, DECODER_DIM, SAR_CHANNELS)
        nt  = sum(p.numel() for p in self.parameters())
        ns  = sum(p.numel() for p in self.scad.parameters())
        np_ = sum(p.numel() for p in self.pmfd.parameters())
        print(_fmt("Total parameters :", f"{nt:,}"))
        print(_fmt("SCAD parameters  :", f"{ns:,}"))
        print(_fmt("PMFD parameters  :", f"{np_:,}"))
        print(_fmt("Backbone strategy:", self.backbone._strategy))
        print(_fmt("Ablation mode    :", f"{'ON  (' + str(len(ABLATION_VARIANTS)) + ' variants)' if ABLATION else 'OFF'}"))
        print(_fmt("Device           :", DEVICE))

    def _ablate_inputs(self, hls, sar, idx, cfg: AblationConfig):
        h, s, x = hls.clone(), sar.clone(), idx.clone()
        if cfg.zero_sar:       s.zero_()
        if cfg.zero_indices:   x.zero_()
        if cfg.zero_nir:       h[:, 4].zero_(); h[:, 5].zero_()
        if cfg.zero_swir:      h[:, 3].zero_(); h[:, 4].zero_()
        if cfg.zero_red_edge:  h[:, 2].zero_()
        return h, s, x

    def _run_variant(self, hls, sar, idx, cfg: AblationConfig) -> ModelOutput:
        B       = hls.shape[0]
        h, s, x = self._ablate_inputs(hls, sar, idx, cfg)
        tokens  = self.backbone(h)
        S_out   = int(math.sqrt(tokens.shape[1])) * (2 ** 4)
        if cfg.disable_scad:
            chl = nitro = torch.zeros(B, 1, S_out, S_out, device=h.device, dtype=h.dtype)
        else:
            chl, nitro = self.scad(tokens, x, disable_cross_attn=cfg.disable_cross_attn)
        if cfg.disable_pmfd:
            bio = loss = torch.zeros(B, 1, S_out, S_out, device=h.device, dtype=h.dtype)
        else:
            bio, loss = self.pmfd(tokens, s, disable_ssm=cfg.disable_ssm, disable_scatter=cfg.disable_scatter)
        return ModelOutput(chl_map=chl, nitro_map=nitro, biomass_map=bio, loss_map=loss, variant_name=cfg.name)

    def forward(self, hls, sar, indices, ablation: bool = ABLATION):
        if not ablation:
            out = self._run_variant(hls, sar, indices, AblationConfig("Full model (baseline)"))
            return out.chl_map, out.nitro_map, out.biomass_map, out.loss_map
        print(f"\n  Running ablation study  ·  {len(ABLATION_VARIANTS)} variants …")
        result = AblationOutput()
        for i, vcfg in enumerate(ABLATION_VARIANTS):
            with torch.no_grad():
                out = self._run_variant(hls, sar, indices, vcfg)
            result.variants[vcfg.name] = out
            print(f"  [{i+1:02d}/{len(ABLATION_VARIANTS)}]  {vcfg.name}")
        print(result.summary_table())
        return result

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True


def build_model(device=DEVICE):
    return DualTaskCropHealthModel().to(device).eval()


def download_hls_chips(n=NUM_TEST_CHIPS, data_dir=DATA_DIR):
    _section("DATASET  ·  ibm-nasa-geospatial / multi-temporal-crop-classification")
    print(_fmt("Archive  :", f"{DATASET_ARCHIVE}  (~1.18 GB)"))
    print(_fmt("Format   :", "224×224 GeoTIFF  |  18 bands  ([B02-B07] × 3 timestamps)"))
    print(_fmt("Labels   :", "USDA Crop Data Layer (CDL)  —  13 crop classes"))
    print(_fmt("Data dir :", data_dir))
    print(f"\n  Downloading {n} validation chips …\n")

    try:
        from huggingface_hub import hf_hub_download
    except ImportError as exc:
        raise RuntimeError("pip install huggingface-hub") from exc
    try:
        import rasterio
    except ImportError as exc:
        raise RuntimeError("pip install rasterio") from exc

    os.makedirs(data_dir, exist_ok=True)

    tgz_path    = hf_hub_download(repo_id=DATASET_REPO, filename=DATASET_ARCHIVE, repo_type="dataset")
    extract_dir = Path(data_dir) / "validation_chips_extracted"

    if not extract_dir.exists():
        print(f"  Extracting → {extract_dir} …")
        extract_dir.mkdir(parents=True, exist_ok=True)
        with tarfile.open(tgz_path, "r:gz") as tar:
            tar.extractall(path=extract_dir)
        print("  Extraction complete.\n")
    else:
        print(f"  Using cached extraction: {extract_dir}\n")

    # Skip macOS AppleDouble resource-fork ghost files (._*)
    merged_paths = sorted(
        p for p in extract_dir.rglob("*_merged.tif")
        if not p.name.startswith("._")
    )
    if not merged_paths:
        raise RuntimeError(f"No valid *_merged.tif files found under {extract_dir}.")

    chips, masks, names = [], [], []
    for i, mp in enumerate(merged_paths[:n]):
        mask_path = Path(str(mp).replace("_merged.tif", "_mask.tif"))
        with rasterio.open(str(mp)) as src:
            data = src.read().astype(np.float32)
        chip = data[:6]
        chips.append(chip)
        if mask_path.exists() and not mask_path.name.startswith("._"):
            with rasterio.open(str(mask_path)) as src:
                masks.append(src.read(1).astype(np.int32))
        else:
            masks.append(None)
        names.append(mp.name)
        print(f"  ✓  [{i+1}/{n}]  {mp.name}  shape={chip.shape}")

    print(f"\n  Loaded {len(chips)} chips  (first timestamp, bands B02–B07)")
    return chips, masks, names


def compute_gt_proxies(chip: np.ndarray) -> dict:
    B02, B03, B04 = chip[0], chip[1], chip[2]
    B05, B06, B07 = chip[3], chip[4], chip[5]
    eps  = 1e-8
    ndvi = (B07 - B04) / (B07 + B04 + eps)
    ndre = (B07 - B05) / (B07 + B05 + eps)
    cire = (B07 / (B05 + eps)) - 1.0
    chl_gt   = np.clip(cire / 10.0, 0, 1) * MAX_CHL_UG_CM2
    n_gt_pct = np.clip(ndre * 2.5 + 0.5, 0, MAX_N_PERCENT)
    agb_gt   = (0.7 * np.clip(ndvi, 0, 1) + 0.3 * (B07 / (B07.max() + eps))) * 120.0
    veg_mask = ndvi > VEG_NDVI_THRESH
    return {"chl_gt_ug_cm2": chl_gt, "n_gt_pct": n_gt_pct, "agb_gt_mgha": agb_gt,
            "ndvi": ndvi, "ndre": ndre, "cire": cire,
            "veg_mask": veg_mask, "veg_pct": 100.0 * veg_mask.sum() / veg_mask.size}


def process_chip(chip_raw, model, device):
    hls      = build_hls_bands(chip_raw)
    H, W     = hls.shape[1], hls.shape[2]
    sar      = build_sar_proxy((H, W))
    indices  = compute_spectral_indices(hls)
    hls_norm = normalize_hls(hls)
    tiles    = tile_image(hls_norm, sar, indices, TILE_SIZE, TILE_STRIDE)
    results  = []

    if ABLATION:
        abl_out = None
        for tile in tiles:
            hls_t = torch.from_numpy(tile["hls"]).float().unsqueeze(0).to(device)
            sar_t = torch.from_numpy(tile["sar"]).float().unsqueeze(0).to(device)
            idx_t = torch.from_numpy(tile["indices"]).float().unsqueeze(0).to(device)
            with torch.no_grad():
                out = model(hls_t, sar_t, idx_t, ablation=True)
            baseline = out.variants["Full model (baseline)"]
            results.append({**tile,
                            "chl_map":     baseline.chl_map.squeeze().cpu().numpy(),
                            "nitro_map":   baseline.nitro_map.squeeze().cpu().numpy(),
                            "biomass_map": baseline.biomass_map.squeeze().cpu().numpy(),
                            "loss_map":    baseline.loss_map.squeeze().cpu().numpy()})
            if abl_out is None:
                abl_out = out
    else:
        for tile in tiles:
            hls_t = torch.from_numpy(tile["hls"]).float().unsqueeze(0).to(device)
            sar_t = torch.from_numpy(tile["sar"]).float().unsqueeze(0).to(device)
            idx_t = torch.from_numpy(tile["indices"]).float().unsqueeze(0).to(device)
            with torch.no_grad():
                c, n, b, l = model(hls_t, sar_t, idx_t, ablation=False)
            results.append({**tile,
                            "chl_map":     c.squeeze().cpu().numpy(),
                            "nitro_map":   n.squeeze().cpu().numpy(),
                            "biomass_map": b.squeeze().cpu().numpy(),
                            "loss_map":    l.squeeze().cpu().numpy()})
        abl_out = None

    maps = stitch_maps(results, hls.shape, TILE_SIZE)
    return maps, hls, sar, indices, abl_out


def stitch_maps(results, image_shape, tile_size=TILE_SIZE):
    _, H, W = image_shape
    acc = {k: np.zeros((H, W), dtype=np.float32)
           for k in ("chlorophyll", "nitrogen", "biomass", "biomass_loss")}
    weight  = np.zeros((H, W), dtype=np.float32)
    key_map = {"chlorophyll": "chl_map", "nitrogen": "nitro_map",
               "biomass": "biomass_map", "biomass_loss": "loss_map"}
    for r in results:
        sy, ey = r["y_start"], r["y_end"]
        sx, ex = r["x_start"], r["x_end"]
        vh, vw = r["valid_h"], r["valid_w"]
        for out_k, in_k in key_map.items():
            patch = r[in_k]
            if patch.shape != (vh, vw):
                patch = cv2.resize(patch, (vw, vh), interpolation=cv2.INTER_LINEAR)
            acc[out_k][sy:ey, sx:ex] += patch
        weight[sy:ey, sx:ex] += 1.0
    w = np.maximum(weight, 1.0)
    return {k: v / w for k, v in acc.items()} | {"weight": weight}


def compute_metrics(maps, indices_full=None):
    chl  = maps["chlorophyll"]
    nit  = maps["nitrogen"]
    bio  = maps["biomass"]
    loss = maps["biomass_loss"]
    H, W = chl.shape
    ndvi     = indices_full[0] if indices_full is not None else np.zeros_like(chl)
    veg_mask = ndvi > VEG_NDVI_THRESH
    total_px = H * W
    stressed_mask = (chl  < STRESS_THRESHOLD)   & veg_mask
    bio_loss_mask = (loss > BIOMASS_LOSS_THRESH) & veg_mask
    veg_pct      = 100.0 * veg_mask.sum() / total_px
    stressed_pct = 100.0 * stressed_mask.sum() / max(veg_mask.sum(), 1)
    bio_loss_pct = 100.0 * bio_loss_mask.sum() / max(veg_mask.sum(), 1)
    def _vm(arr): return float(arr[veg_mask].mean()) if veg_mask.any() else 0.0
    mean_chl_ug   = _vm(chl  * MAX_CHL_UG_CM2)
    mean_n_pct    = _vm(nit  * MAX_N_PERCENT)
    mean_agb      = _vm(bio  * MAX_BIOMASS_MGHA)
    mean_loss_agb = _vm(loss * HEALTHY_BIOMASS_REF)
    mean_chl_raw  = _vm(chl)
    severity = "SEVERE" if stressed_pct > 50 else "MODERATE" if stressed_pct > 20 else "MILD"
    return {
        "image_size_px":           [H, W],
        "total_pixels":            total_px,
        "vegetation_coverage_%":   round(veg_pct, 2),
        "stressed_area_%_of_veg":  round(stressed_pct, 2),
        "biomass_loss_area_%_veg": round(bio_loss_pct, 2),
        "chlorophyll_ug_cm2_mean": round(mean_chl_ug, 3),
        "chlorophyll_%_of_healthy":round(min(100.0, 100.0 * mean_chl_ug / (MAX_CHL_UG_CM2 + 1e-8)), 2),
        "chlorophyll_stress_%":    round((1 - mean_chl_raw) * 100, 2),
        "N_concentration_%":       round(mean_n_pct, 3),
        "N_normalized_0to100_%":   round(min(100.0, 100.0 * mean_n_pct / (MAX_N_PERCENT + 1e-8)), 2),
        "biomass_AGB_Mgha_mean":   round(mean_agb, 2),
        "biomass_%_of_max":        round(min(100.0, 100.0 * mean_agb / (MAX_BIOMASS_MGHA + 1e-8)), 2),
        "biomass_loss_Mgha_mean":  round(mean_loss_agb, 2),
        "biomass_loss_%":          round(min(100.0, 100.0 * mean_loss_agb / (HEALTHY_BIOMASS_REF + 1e-8)), 2),
        "stress_severity":         severity,
    }


def compute_error_metrics(pred_map, gt_map, mask):
    if not mask.any():
        return {"mae": float("nan"), "rmse": float("nan"), "r": float("nan"), "bias_%": float("nan")}
    p    = pred_map[mask].ravel()
    g    = gt_map[mask].ravel()
    mae  = float(np.abs(p - g).mean())
    rmse = float(np.sqrt(((p - g) ** 2).mean()))
    r    = float(np.corrcoef(p, g)[0, 1]) if p.std() > 0 and g.std() > 0 else float("nan")
    bias = float((p - g).mean() / (g.mean() + 1e-8) * 100)
    return {"mae": mae, "rmse": rmse, "r": r, "bias_%": bias}


def print_chip_result(name, idx, gt, pred_m, err_chl, err_bio, elapsed):
    chl_pred_ug = pred_m["chlorophyll_%_of_healthy"] * MAX_CHL_UG_CM2 / 100
    chl_gt_m    = float(gt["chl_gt_ug_cm2"][gt["veg_mask"]].mean()) if gt["veg_mask"].any() else 0
    bio_gt_m    = float(gt["agb_gt_mgha"][gt["veg_mask"]].mean())   if gt["veg_mask"].any() else 0
    r_chl = f"{err_chl['r']:.3f}" if not math.isnan(err_chl["r"]) else "N/A"
    r_bio = f"{err_bio['r']:.3f}" if not math.isnan(err_bio["r"]) else "N/A"
    print(f"\n  ┌─ Chip {idx+1}  ·  {name}")
    print(f"  │  Vegetation coverage    : {gt['veg_pct']:.1f}%")
    print(f"  │")
    print(f"  │  ── Chlorophyll ──────────────────────────────────")
    print(f"  │  GT  (CIre proxy)       : {chl_gt_m:.2f} μg/cm²")
    print(f"  │  Pred (SCAD)            : {chl_pred_ug:.2f} μg/cm²")
    print(f"  │  MAE / RMSE             : {err_chl['mae']:.4f}  /  {err_chl['rmse']:.4f}")
    print(f"  │  Pearson r / Bias       : {r_chl}  /  {err_chl['bias_%']:.1f}%")
    print(f"  │  Stress level           : {pred_m['chlorophyll_stress_%']:.1f}%  [{pred_m['stress_severity']}]")
    print(f"  │")
    print(f"  │  ── Biomass ───────────────────────────────────────")
    print(f"  │  GT  (NDVI-AGB proxy)  : {bio_gt_m:.1f} Mg/ha")
    print(f"  │  Pred (PMFD)           : {pred_m['biomass_AGB_Mgha_mean']:.1f} Mg/ha")
    print(f"  │  MAE / RMSE            : {err_bio['mae']:.2f}  /  {err_bio['rmse']:.2f}")
    print(f"  │  Pearson r / Bias      : {r_bio}  /  {err_bio['bias_%']:.1f}%")
    print(f"  │  Biomass loss area     : {pred_m['biomass_loss_area_%_veg']:.1f}% of veg")
    print(f"  │  N concentration       : {pred_m['N_concentration_%']:.3f}%")
    print(f"  └─ Inference time        : {elapsed:.2f}s")


def print_aggregate_table(rows):
    _section("AGGREGATE TEST RESULTS")
    hdr = (f"  {'Chip':<22} {'Veg%':>6} {'ChlStr%':>8} {'ChlGT':>9} {'ChlPr':>9}"
           f" {'N%':>6} {'AGB_GT':>7} {'AGB_Pr':>7} {'BioLoss%':>9} {'Sev':>8}  {'t(s)':>5}")
    sep = "  " + "─" * (len(hdr) - 2)
    print(hdr); print(sep)
    for r in rows:
        print(f"  {r['name'][:20]:<22} {r['veg_cov_%']:>5.1f}% {r['chl_stress_%']:>7.1f}%"
              f" {r['chl_gt_ug']:>8.2f} {r['chl_pred_ug']:>8.2f}"
              f" {r['n_pct']:>5.3f}% {r['bio_gt_mh']:>6.1f} {r['bio_pred_mh']:>6.1f}"
              f" {r['bio_loss_%']:>8.1f}% {r['severity']:>8}  {r['time_s']:>4.1f}s")
    print(sep)
    avg = lambda k: np.nanmean([r[k] for r in rows])
    print(f"\n  Chlorophyll  MAE : {avg('chl_mae'):.4f} μg/cm²   RMSE : {avg('chl_rmse'):.4f}   r : {avg('chl_r'):.3f}")
    print(f"  Biomass      MAE : {avg('bio_mae'):.2f} Mg/ha    RMSE : {avg('bio_rmse'):.2f}    r : {avg('bio_r'):.3f}")
    total_t = sum(r["time_s"] for r in rows)
    print(f"  Total time       : {total_t:.1f}s   ({total_t/len(rows):.2f}s / chip)")


def render_chip_panel(chips_raw, maps_list, gt_list, names, output_dir):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    n, cols = len(chips_raw), 5
    fig  = plt.figure(figsize=(cols * 3.8, n * 3.6), facecolor="#0d1117")
    fig.suptitle("Crop Health Monitor  ·  Prithvi-EO-1.0-100M  (SCAD + PMFD)",
                 color="white", fontsize=13, fontweight="bold", y=0.995)
    col_titles = ["False-Color RGB\n(B07-B04-B03)", "Chlorophyll Stress\n(SCAD)",
                  "N Concentration\n(SCAD)", "Biomass Density\n(PMFD)", "Biomass Loss\n(PMFD)"]
    def norm_ch(c):
        lo, hi = np.percentile(c, 2), np.percentile(c, 98)
        return np.clip((c - lo) / (hi - lo + 1e-8), 0, 1)
    axes = [[fig.add_subplot(n, cols, r * cols + c + 1) for c in range(cols)] for r in range(n)]
    for ci in range(n):
        chip, maps, gt, name = chips_raw[ci], maps_list[ci], gt_list[ci], names[ci]
        H, W = maps["chlorophyll"].shape
        rgb_vis = np.stack([norm_ch(chip[5][:H, :W]), norm_ch(chip[2][:H, :W]),
                            norm_ch(chip[1][:H, :W])], axis=-1)
        veg     = gt["ndvi"][:H, :W] > VEG_NDVI_THRESH
        veg_ov  = np.zeros((*rgb_vis.shape[:2], 4))
        veg_ov[veg] = [0.1, 0.8, 0.1, 0.22]
        lbl = name.replace("_merged.tif", "").replace("_", " ")
        axes[ci][0].imshow(rgb_vis); axes[ci][0].imshow(veg_ov)
        axes[ci][0].set_title(lbl, color="#aaaaaa", fontsize=7.5, pad=2)
        for j, (key, cmap) in enumerate(
            zip(["chlorophyll", "nitrogen", "biomass", "biomass_loss"],
                ["RdYlGn_r", "PuBuGn", "YlOrBr", "RdPu"]), start=1):
            im = axes[ci][j].imshow(maps[key][:H, :W], cmap=cmap, vmin=0, vmax=1)
            fig.colorbar(im, ax=axes[ci][j], fraction=0.045, pad=0.02).ax.tick_params(
                labelcolor="white", labelsize=6)
        for c in range(cols):
            axes[ci][c].set_xticks([]); axes[ci][c].set_yticks([])
            for sp in axes[ci][c].spines.values(): sp.set_edgecolor("#333344")
    for c, t in enumerate(col_titles):
        axes[0][c].set_title(t, color="#e0e0ff", fontsize=8.5, fontweight="bold", pad=4)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    os.makedirs(output_dir, exist_ok=True)
    out = os.path.join(output_dir, "panel_maps.png")
    plt.savefig(out, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"  ✓  Panel saved  →  {os.path.abspath(out)}")


def render_metrics_bar_chart(rows, output_dir):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mtick
    names      = [r["name"][:18]    for r in rows]
    chl_stress = [r["chl_stress_%"] for r in rows]
    bio_loss   = [r["bio_loss_%"]   for r in rows]
    veg_cov    = [r["veg_cov_%"]    for r in rows]
    n_level    = [r["n_norm_%"]     for r in rows]
    x, w       = np.arange(len(names)), 0.20
    fig, ax = plt.subplots(figsize=(max(10, 2.2 * len(names)), 5), facecolor="#0d1117")
    ax.set_facecolor("#0d1117")
    ax.bar(x - 1.5*w, veg_cov,    w, label="Veg Coverage %", color="#2ecc71", alpha=0.85)
    ax.bar(x - 0.5*w, chl_stress, w, label="Chl Stress %",   color="#e74c3c", alpha=0.85)
    ax.bar(x + 0.5*w, n_level,    w, label="N Level %",       color="#3498db", alpha=0.85)
    ax.bar(x + 1.5*w, bio_loss,   w, label="Biomass Loss %",  color="#e67e22", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=20, ha="right", color="#cccccc", fontsize=8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.tick_params(colors="#aaaaaa")
    ax.set_ylabel("Percentage (%)", color="#aaaaaa")
    ax.set_title("Per-Chip Metrics  ·  Prithvi-EO-1.0-100M + SCAD + PMFD",
                 color="white", fontweight="bold", fontsize=11)
    ax.legend(loc="upper right", facecolor="#1a1a2e", edgecolor="#555555",
              labelcolor="white", fontsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor("#333344")
    ax.grid(axis="y", color="#222233", linewidth=0.6)
    plt.tight_layout()
    out = os.path.join(output_dir, "metrics_bar.png")
    plt.savefig(out, dpi=120, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"  ✓  Bar chart saved  →  {os.path.abspath(out)}")


def render_correlation_scatter(rows, output_dir):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    chl_pred = [r["chl_pred_ug"] for r in rows]
    chl_gt   = [r["chl_gt_ug"]   for r in rows]
    bio_pred = [r["bio_pred_mh"] for r in rows]
    bio_gt   = [r["bio_gt_mh"]   for r in rows]
    colors   = plt.cm.plasma(np.linspace(0.2, 0.9, len(chl_pred)))
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), facecolor="#0d1117")
    for ax in axes:
        ax.set_facecolor("#111122")
        for sp in ax.spines.values(): sp.set_edgecolor("#333344")
        ax.tick_params(colors="#aaaaaa")
    panels = [
        (axes[0], chl_pred, chl_gt,
         "Chlorophyll: Pred vs GT", "GT Chlorophyll (μg/cm²)", "Pred Chlorophyll (μg/cm²) — SCAD"),
        (axes[1], bio_pred, bio_gt,
         "Biomass: Pred vs GT", "GT Biomass (Mg/ha)", "Pred Biomass (Mg/ha) — PMFD"),
    ]
    for ax, pred, gt, title, xlabel, ylabel in panels:
        ax.scatter(gt, pred, c=colors, s=90, zorder=5, edgecolors="white", linewidths=0.6)
        lim = max(max(gt + pred) * 1.1, 1)
        ax.plot([0, lim], [0, lim], "--", color="#666688", lw=1.2, label="1:1 line")
        ax.set_xlabel(xlabel, color="#aaaaaa")
        ax.set_ylabel(ylabel, color="#aaaaaa")
        ax.set_title(title, color="white", fontweight="bold")
        ax.legend(facecolor="#1a1a2e", edgecolor="#555555", labelcolor="white")
        if len(pred) > 1:
            try:
                r_val = np.corrcoef(gt, pred)[0, 1]
                ax.annotate(f"r = {r_val:.3f}", xy=(0.05, 0.88), xycoords="axes fraction",
                            color="#00ff88", fontsize=11, fontweight="bold")
            except Exception:
                pass
    plt.suptitle("Pred vs Spectral GT Proxies", color="white", fontweight="bold", fontsize=12)
    plt.tight_layout()
    out = os.path.join(output_dir, "correlation_scatter.png")
    plt.savefig(out, dpi=120, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"  ✓  Scatter saved  →  {os.path.abspath(out)}")


# if __name__ == "__main__":
#     _section(f"DualTask Crop Health Monitor  ·  device={DEVICE}  ·  ablation={'ON' if ABLATION else 'OFF'}")

#     chips_raw, masks, names = download_hls_chips(NUM_TEST_CHIPS, DATA_DIR)
#     n_loaded = len(chips_raw)

#     model = build_model(DEVICE)
#     model.eval()

#     _section(f"INFERENCE  ·  {n_loaded} chips  ·  ablation={'ON' if ABLATION else 'OFF'}")
#     rows, all_chips_raw, all_maps, all_gts = [], [], [], []

#     for ci, (chip_raw, mask, name) in enumerate(zip(chips_raw, masks, names)):
#         print(f"\n  ── Chip {ci+1}/{n_loaded}  ·  {name}")
#         t0  = time.time()
#         gt  = compute_gt_proxies(chip_raw)
#         maps, hls, sar, indices, abl_out = process_chip(chip_raw, model, DEVICE)
#         elapsed = time.time() - t0

#         H, W   = maps["chlorophyll"].shape
#         vm     = gt["veg_mask"][:H, :W]
#         pred_m = compute_metrics(maps, indices)

#         chl_pred_n = maps["chlorophyll"][:H, :W] * MAX_CHL_UG_CM2
#         bio_pred_n = maps["biomass"][:H, :W]     * MAX_BIOMASS_MGHA
#         err_chl    = compute_error_metrics(chl_pred_n, gt["chl_gt_ug_cm2"][:H, :W], vm)
#         err_bio    = compute_error_metrics(bio_pred_n, gt["agb_gt_mgha"][:H, :W],   vm)

#         print_chip_result(name, ci, gt, pred_m, err_chl, err_bio, elapsed)

#         chl_gt_m   = float(gt["chl_gt_ug_cm2"][gt["veg_mask"]].mean()) if gt["veg_mask"].any() else 0.0
#         bio_gt_m   = float(gt["agb_gt_mgha"][gt["veg_mask"]].mean())   if gt["veg_mask"].any() else 0.0
#         chl_pred_m = pred_m["chlorophyll_%_of_healthy"] * MAX_CHL_UG_CM2 / 100
#         bio_pred_m = pred_m["biomass_AGB_Mgha_mean"]

#         rows.append({
#             "name":         name,
#             "veg_cov_%":    pred_m["vegetation_coverage_%"],
#             "chl_stress_%": pred_m["chlorophyll_stress_%"],
#             "chl_gt_ug":    chl_gt_m,
#             "chl_pred_ug":  chl_pred_m,
#             "n_pct":        pred_m["N_concentration_%"],
#             "n_norm_%":     pred_m["N_normalized_0to100_%"],
#             "bio_gt_mh":    bio_gt_m,
#             "bio_pred_mh":  bio_pred_m,
#             "bio_loss_%":   pred_m["biomass_loss_%"],
#             "severity":     pred_m["stress_severity"],
#             "chl_mae":      err_chl["mae"],
#             "chl_rmse":     err_chl["rmse"],
#             "chl_r":        err_chl["r"] if not math.isnan(err_chl["r"]) else 0.0,
#             "bio_mae":      err_bio["mae"],
#             "bio_rmse":     err_bio["rmse"],
#             "bio_r":        err_bio["r"] if not math.isnan(err_bio["r"]) else 0.0,
#             "time_s":       elapsed,
#         })
#         all_chips_raw.append(chip_raw)
#         all_maps.append(maps)
#         all_gts.append(gt)

#     print_aggregate_table(rows)

#     _section("VISUALIZATIONS")
#     os.makedirs(OUTPUT_DIR, exist_ok=True)
#     render_chip_panel(all_chips_raw, all_maps, all_gts, names, OUTPUT_DIR)
#     render_metrics_bar_chart(rows, OUTPUT_DIR)
#     render_correlation_scatter(rows, OUTPUT_DIR)

#     _section("DONE")
#     print(_fmt("Chips tested  :", n_loaded))
#     print(_fmt("Output folder :", os.path.abspath(OUTPUT_DIR)))
#     print(f"  ├── panel_maps.png")
#     print(f"  ├── metrics_bar.png")
#     print(f"  └── correlation_scatter.png")
#     print(f"\n  Dataset : Cecil et al. (2023)  doi:10.57967/hf/0955")
#     print(f"  Model   : Jakubik et al. (2023) arXiv:2310.18660")
#     print("━" * 72)

In [2]:
!pip install torch torchvision rasterio huggingface_hub transformers
!pip install einops timm 

In [3]:
import os
import sys
import math
import time
import tarfile
import warnings
from pathlib import Path
from typing import List, Tuple, Optional

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# sys.path.insert(0, os.path.dirname(os.path.abspath(__file__) if "__file__" in dir() else os.getcwd()))
# from crop_health_monitor import (
#     DualTaskCropHealthModel,
#     build_hls_bands, normalize_hls, build_sar_proxy, compute_spectral_indices,
#     HLS_MEAN, HLS_STD,
#     MAX_CHL_UG_CM2, MAX_N_PERCENT, MAX_BIOMASS_MGHA, HEALTHY_BIOMASS_REF,
#     VEG_NDVI_THRESH, TILE_SIZE, DEVICE, DATASET_REPO,
#     _section, _fmt,
# )

TRAIN_ARCHIVE  = "training_chips.tgz"
VAL_ARCHIVE    = "validation_chips.tgz"
DATA_DIR       = "./data"
OUTPUT_DIR     = "./output"
CKPT_PATH      = os.path.join(OUTPUT_DIR, "trained_model.pt")
BEST_CKPT_PATH = os.path.join(OUTPUT_DIR, "best_model.pt")

EPOCHS         = 30
BATCH_SIZE     = 16
LR_DECODERS    = 3e-4
LR_BACKBONE    = 3e-5
UNFREEZE_EPOCH = 5       # backbone unfreezes after this many epochs
WEIGHT_DECAY   = 1e-4
NUM_WORKERS    = 2
VAL_EVERY      = 1
GRAD_CLIP      = 1.0

W_CHL    = 1.0
W_NITRO  = 1.0
W_BIO    = 1.0
W_LOSS   = 0.8


def _fmt(label, value, width=38):
    return f"  {label:<{width}}{value}"

def _section(title):
    w = 72
    print(f"\n{'━'*w}\n  {title}\n{'━'*w}")


def download_chips(archive: str, data_dir: str = DATA_DIR) -> Path:
    from huggingface_hub import hf_hub_download
    os.makedirs(data_dir, exist_ok=True)
    tgz = hf_hub_download(repo_id=DATASET_REPO, filename=archive, repo_type="dataset")
    extract_dir = Path(data_dir) / (archive.replace(".tgz", "_extracted"))
    if not extract_dir.exists():
        print(f"  Extracting {archive} → {extract_dir} …")
        extract_dir.mkdir(parents=True, exist_ok=True)
        with tarfile.open(tgz, "r:gz") as tar:
            tar.extractall(path=extract_dir)
        print("  Done.\n")
    else:
        print(f"  Cached: {extract_dir}\n")
    return extract_dir


def load_chip_paths(extract_dir: Path) -> List[Tuple[Path, Optional[Path]]]:
    merged = sorted(p for p in extract_dir.rglob("*_merged.tif") if not p.name.startswith("._"))
    pairs  = []
    for mp in merged:
        mask_p = Path(str(mp).replace("_merged.tif", "_mask.tif"))
        pairs.append((mp, mask_p if mask_p.exists() and not mask_p.name.startswith("._") else None))
    return pairs


def compute_pseudo_gt(chip: np.ndarray) -> dict:
    B02, B03, B04 = chip[0], chip[1], chip[2]
    B05, B06, B07 = chip[3], chip[4], chip[5]
    eps  = 1e-8
    ndvi = (B07 - B04) / (B07 + B04 + eps)
    ndre = (B07 - B05) / (B07 + B05 + eps)
    cire = (B07 / (B05 + eps)) - 1.0
    chl  = np.clip(cire / 10.0, 0, 1)
    nit  = np.clip(ndre * 2.5 + 0.5, 0, MAX_N_PERCENT) / MAX_N_PERCENT
    ndvi_c = np.clip(ndvi, 0, 1)
    bio  = np.clip((0.7 * ndvi_c + 0.3 * (B07 / (B07.max() + eps))) * 120.0 / MAX_BIOMASS_MGHA, 0, 1)
    loss = np.clip(1.0 - ndvi_c, 0, 1)
    return {"chl": chl, "nit": nit, "bio": bio, "loss": loss}


class ChipDataset(Dataset):
    def __init__(self, pairs: List[Tuple[Path, Optional[Path]]], augment: bool = True):
        import rasterio
        self.pairs   = pairs
        self.augment = augment
        self._rio    = rasterio

    def __len__(self):
        return len(self.pairs)

    def _augment(self, *arrays):
        if np.random.rand() > 0.5:
            arrays = tuple(np.flip(a, axis=-1).copy() for a in arrays)
        if np.random.rand() > 0.5:
            arrays = tuple(np.flip(a, axis=-2).copy() for a in arrays)
        k = np.random.randint(0, 4)
        if k:
            arrays = tuple(np.rot90(a, k=k, axes=(-2, -1)).copy() for a in arrays)
        return arrays

    def __getitem__(self, idx):
        mp, _ = self.pairs[idx]
        with self._rio.open(str(mp)) as src:
            data = src.read().astype(np.float32)

        chip     = data[:6]
        hls      = build_hls_bands(chip)
        sar      = build_sar_proxy((hls.shape[1], hls.shape[2]))
        indices  = compute_spectral_indices(hls)
        hls_norm = normalize_hls(hls)
        gt       = compute_pseudo_gt(chip)

        gt_stack = np.stack([gt["chl"], gt["nit"], gt["bio"], gt["loss"]])

        if self.augment:
            hls_norm, sar, indices, gt_stack = self._augment(hls_norm, sar, indices, gt_stack)

        return (
            torch.from_numpy(hls_norm).float(),
            torch.from_numpy(sar).float(),
            torch.from_numpy(indices).float(),
            torch.from_numpy(gt_stack).float(),
        )


def biophysical_loss(
    chl_pred, nitro_pred, bio_pred, loss_pred,
    gt: torch.Tensor,
    veg_weight: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    def _mse(p, g):
        diff = (p.squeeze(1) - g) ** 2
        if veg_weight is not None:
            diff = diff * veg_weight
        return diff.mean()

    chl_gt, nit_gt, bio_gt, los_gt = gt[:, 0], gt[:, 1], gt[:, 2], gt[:, 3]
    return (
        W_CHL   * _mse(chl_pred,   chl_gt)  +
        W_NITRO * _mse(nitro_pred, nit_gt)  +
        W_BIO   * _mse(bio_pred,   bio_gt)  +
        W_LOSS  * _mse(loss_pred,  los_gt)
    )


def _resize_preds(chl, nit, bio, los, target_h, target_w):
    def _r(t):
        if t.shape[-2] == target_h and t.shape[-1] == target_w:
            return t
        return F.interpolate(t, (target_h, target_w), mode="bilinear", align_corners=False)
    return _r(chl), _r(nit), _r(bio), _r(los)


def run_epoch(model, loader, optimizer, device, train: bool) -> dict:
    model.train() if train else model.eval()
    total_loss = 0.0
    n_batches  = 0
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        for hls, sar, idx, gt in loader:
            hls, sar, idx, gt = hls.to(device), sar.to(device), idx.to(device), gt.to(device)

            chl, nit, bio, los = model(hls, sar, idx, ablation=False)

            H, W = gt.shape[-2], gt.shape[-1]
            chl, nit, bio, los = _resize_preds(chl, nit, bio, los, H, W)

            ndvi       = idx[:, 0]
            veg_w      = (ndvi > VEG_NDVI_THRESH).float() * 1.5 + 0.5
            loss_val   = biophysical_loss(chl, nit, bio, los, gt, veg_w)

            if train:
                optimizer.zero_grad()
                loss_val.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            total_loss += loss_val.item()
            n_batches  += 1

    return {"loss": total_loss / max(n_batches, 1)}


def build_optimizer(model, epoch: int):
    backbone_params = list(model.backbone.parameters())
    decoder_params  = list(model.scad.parameters()) + list(model.pmfd.parameters())
    if epoch < UNFREEZE_EPOCH:
        for p in backbone_params:
            p.requires_grad = False
        groups = [{"params": decoder_params, "lr": LR_DECODERS}]
    else:
        for p in backbone_params:
            p.requires_grad = True
        groups = [
            {"params": decoder_params,  "lr": LR_DECODERS},
            {"params": backbone_params, "lr": LR_BACKBONE},
        ]
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def print_epoch(epoch, epochs, phase, metrics, elapsed, frozen):
    status = "backbone=frozen" if frozen else "backbone=active"
    print(f"  [{epoch:>3}/{epochs}]  {phase:<5}  loss={metrics['loss']:.5f}"
          f"  {status}  t={elapsed:.1f}s")


if __name__ == "__main__":
    _section(f"CROP HEALTH TRAINING  ·  device={DEVICE}")
    print(_fmt("Backbone :", "Prithvi-EO-1.0-100M  (frozen first, then fine-tuned)"))
    print(_fmt("Decoders :", "SCAD + PMFD  (random init → trained on pseudo-labels)"))
    print(_fmt("GT source:", "Spectral proxies: CIre→CHL  NDRE→N  NDVI→AGB"))
    print(_fmt("Epochs   :", EPOCHS))
    print(_fmt("Batch    :", BATCH_SIZE))
    print(_fmt("LR decod :", LR_DECODERS))
    print(_fmt("LR backb :", f"{LR_BACKBONE}  (active from epoch {UNFREEZE_EPOCH+1})"))
    print(_fmt("Output   :", CKPT_PATH))

    _section("DOWNLOADING DATA")
    # train_dir = download_chips(TRAIN_ARCHIVE, DATA_DIR)
    # val_dir   = download_chips(VAL_ARCHIVE,   DATA_DIR)

    train_dir = "/kaggle/input/notebooks/anidiptapal/multi-temporal-crop-classification/data/training_chips_extracted"
    val_dir   =  "/kaggle/input/notebooks/anidiptapal/multi-temporal-crop-classification/data/validation_chips_extracted"
    
    from pathlib import Path
    
    train_pairs = load_chip_paths(Path(train_dir))
    val_pairs   = load_chip_paths(Path(val_dir))
    
    print(f"  Train chips : {len(train_pairs)}")
    print(f"  Val chips   : {len(val_pairs)}")

    train_ds = ChipDataset(train_pairs, augment=True)
    val_ds   = ChipDataset(val_pairs,   augment=False)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

    _section("MODEL INIT")
    model = DualTaskCropHealthModel().to(DEVICE)
    model.freeze_backbone()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    best_val_loss = float("inf")
    history       = []

    _section(f"TRAINING  ·  {EPOCHS} epochs")
    print(f"  {'Epoch':>5}  {'Phase':<5}  {'Loss':>10}  {'Status':<22}  {'Time':>6}")
    print(f"  {'─'*58}")

    for epoch in range(1, EPOCHS + 1):
        frozen = epoch <= UNFREEZE_EPOCH

        if epoch == UNFREEZE_EPOCH + 1:
            print(f"\n  ── Epoch {epoch}: unfreezing backbone  (lr={LR_BACKBONE})\n")
            model.unfreeze_backbone()

        optimizer = build_optimizer(model, epoch)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

        t0         = time.time()
        train_met  = run_epoch(model, train_dl, optimizer, DEVICE, train=True)
        t_train    = time.time() - t0

        print_epoch(epoch, EPOCHS, "train", train_met, t_train, frozen)

        if epoch % VAL_EVERY == 0:
            t0      = time.time()
            val_met = run_epoch(model, val_dl, None, DEVICE, train=False)
            t_val   = time.time() - t0
            print_epoch(epoch, EPOCHS, "val", val_met, t_val, frozen)

            if val_met["loss"] < best_val_loss:
                best_val_loss = val_met["loss"]
                torch.save(model.state_dict(), BEST_CKPT_PATH)
                print(f"  ✓  Best model saved  (val_loss={best_val_loss:.5f})")

            history.append({"epoch": epoch, "train_loss": train_met["loss"],
                            "val_loss": val_met["loss"]})

        scheduler.step()

    torch.save(model.state_dict(), CKPT_PATH)

    _section("TRAINING COMPLETE")
    print(_fmt("Final checkpoint  :", CKPT_PATH))
    print(_fmt("Best checkpoint   :", BEST_CKPT_PATH))
    print(_fmt("Best val loss     :", f"{best_val_loss:.5f}"))
    if history:
        best_ep = min(history, key=lambda r: r["val_loss"])
        print(_fmt("Best epoch        :", best_ep["epoch"]))
    print(f"\n  Load for inference:")
    print(f"    model = build_model(DEVICE)")
    print(f"    model.load_state_dict(torch.load('{BEST_CKPT_PATH}'))")
    print(f"    model.eval()")
    print("━" * 72)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  CROP HEALTH TRAINING  ·  device=cuda
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Backbone :                            Prithvi-EO-1.0-100M  (frozen first, then fine-tuned)
  Decoders :                            SCAD + PMFD  (random init → trained on pseudo-labels)
  GT source:                            Spectral proxies: CIre→CHL  NDRE→N  NDVI→AGB
  Epochs   :                            30
  Batch    :                            16
  LR decod :                            0.0003
  LR backb :                            3e-05  (active from epoch 6)
  Output   :                            ./output/trained_model.pt

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  DOWNLOADING DATA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Train chips : 3083
  Val chips   : 771

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

  [Backbone] ✗ AutoModel failed: 'NoneType' object cannot be interpreted as an integer
  [Backbone] Trying hf_hub_download (Prithvi_EO_V1_100M.pt) …


Prithvi_EO_V1_100M.pt:   0%|          | 0.00/454M [00:00<?, ?B/s]

prithvi_mae.py: 0.00B [00:00, ?B/s]

  [Backbone] Instantiated PrithviViT
  [Backbone] ✓ PrithviViT ready  (86,237,184 params)  missing=150  unexpected=252
  Total parameters :                    108,042,072
  SCAD parameters  :                    15,734,338
  PMFD parameters  :                    6,070,550
  Backbone strategy:                    pt_direct
  Ablation mode    :                    ON  (11 variants)
  Device           :                    cuda

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  TRAINING  ·  30 epochs
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Epoch  Phase        Loss  Status                    Time
  ──────────────────────────────────────────────────────────
  [  1/30]  train  loss=0.50943  backbone=frozen  t=128.4s
  [  1/30]  val    loss=0.28109  backbone=frozen  t=18.1s
  ✓  Best model saved  (val_loss=0.28109)
  [  2/30]  train  loss=0.17632  backbone=frozen  t=133.6s
  [  2/30]  val    loss=0.11498  backbone=frozen  t=12.6s
  ✓  Be